In [1]:
# ─── PHASE 0: Initialize PySpark + Sedona ────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, expr, col
from pyspark.sql.types import StringType, DoubleType, ArrayType, StructType, StructField
from sedona.spark import SedonaContext

os.makedirs("/tmp/ivy2", exist_ok=True)

config = (
    SedonaContext.builder()
    .appName("WBI_Prototype")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.sedona:sedona-spark-3.5_2.12:1.6.1,"
        "org.datasyslab:geotools-wrapper:1.6.1-28.2"
    )
    .config("spark.jars.ivy", "/tmp/ivy2")
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator",
            "org.apache.sedona.core.serde.SedonaKryoRegistrator")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

sedona = SedonaContext.create(config)
spark  = sedona

print("✅ Sedona session started")
print(f"   Spark version : {spark.version}")

✅ Sedona session started
   Spark version : 3.5.0


In [2]:
#PHASE 1.5: Exploratory Data Analysis (EDA)


raw_data = spark.read.csv(
    "/home/jovyan/work/data/AB_NYC_2019.csv",
    header=True,
    inferSchema=True,
    quote='"',        
    escape='"',      
    multiLine=True    
)

print("1. THE SCHEMA (Columns and Data Types):")
raw_data.printSchema()

print("\n2. THE FIRST 3 ROWS:")
raw_data.select("id", "name", "neighbourhood_group", "neighbourhood", "latitude", "longitude", "price").show(3, truncate=False)

print("\n3. STATISTICAL SUMMARY OF 'PRICE':")
raw_data.describe("price").show()

1. THE SCHEMA (Columns and Data Types):
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- last_review: date (nullable = true)
 |-- reviews_per_month: double (nullable = true)
 |-- calculated_host_listings_count: integer (nullable = true)
 |-- availability_365: integer (nullable = true)


2. THE FIRST 3 ROWS:
+----+-----------------------------------+-------------------+-------------+--------+---------+-----+
|id  |name                               |neighbourhood_group|neighbourhood|latitude|longitude|price|
+----+---------------

In [3]:
# ─── PHASE 1: Load house price data ──────────────────────────────────────────

from pyspark.sql.types import DoubleType

# --- 1a. Read raw CSV with strict parsing rules
listings_raw = spark.read.csv(
    "/home/jovyan/work/data/AB_NYC_2019.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

# --- 1b. Select only the columns we need
listings = listings_raw.select(
    "id", "latitude", "longitude", "price",
    "neighbourhood", "neighbourhood_group"
)

# --- 1c. Cast price to DoubleType
listings = listings.withColumn("price", col("price").cast(DoubleType()))

# --- 1d. Drop rows where price is 0 (errors) or coordinates are missing
listings = listings.filter(
    col("price").isNotNull() &
    col("latitude").isNotNull() &
    col("longitude").isNotNull() &
    (col("price") > 0)
)

# --- 1e. Create Sedona geometry column
# IMPORTANT: ST_Point takes (longitude, latitude) — x first, then y.
listings = listings.withColumn(
    "geom",
    expr("ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE))")
)

print(f" Loaded {listings.count():,} clean listings")
listings.select("id", "latitude", "longitude", "price", "geom").show(3, truncate=False)

 Loaded 48,884 clean listings
+----+--------+---------+-----+--------------------------+
|id  |latitude|longitude|price|geom                      |
+----+--------+---------+-----+--------------------------+
|2539|40.64749|-73.97237|149.0|POINT (-73.97237 40.64749)|
|2595|40.75362|-73.98377|225.0|POINT (-73.98377 40.75362)|
|3647|40.80902|-73.9419 |150.0|POINT (-73.9419 40.80902) |
+----+--------+---------+-----+--------------------------+
only showing top 3 rows



In [4]:
# ─── PHASE 2: Download POI data via OSMnx ────────────────────────────────────

import osmnx as ox
import geopandas as gpd
import pandas as pd
from pyspark.sql.functions import expr

print("Downloading POIs from OpenStreetMap")

poi_tags = {
    "amenity": ["pub", "church", "stadium"],
    "leisure": ["fitness_centre"]   
}

# Download data for Manhattan
pois_gdf = ox.features_from_place(
    "Manhattan, New York City, New York, USA",
    tags=poi_tags
)

# Reset index to flatten the data
pois_gdf = pois_gdf.reset_index(drop=True)

# Convert any building polygons into a single center coordinate
pois_gdf["geometry"] = pois_gdf["geometry"].centroid
pois_gdf["lat"]      = pois_gdf["geometry"].y
pois_gdf["lon"]      = pois_gdf["geometry"].x

# Helper function to categorize the POI correctly
def get_category(row):
    if pd.notna(row.get("amenity")): return str(row["amenity"])
    if pd.notna(row.get("leisure")): return "gym"
    return "unknown"

pois_gdf["category"] = pois_gdf.apply(get_category, axis=1)

# Assign the well-being weights
weight_map = {
    "gym":      0.8,   # health infrastructure
    "church":   0.5,   # community gathering
    "stadium":  0.7,   # public space and events
    "pub":      0.4    # social space
}
pois_gdf["poi_weight"] = pois_gdf["category"].map(weight_map).fillna(0.4)

pois_clean = pois_gdf[["lat", "lon", "category", "poi_weight"]].dropna(subset=["lat", "lon"])

# --- Convert the Pandas DataFrame into a distributed PySpark DataFrame ---
poi_spark = spark.createDataFrame(pois_clean)

# Add the Sedona geometry column (remember: longitude first!)
poi_spark = poi_spark.withColumn(
    "geom",
    expr("ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE))")
)

print(f" Loaded {poi_spark.count():,} POIs into Spark")
poi_spark.groupBy("category").count().show()

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


/tmp/ipykernel_8261/68245619.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois_gdf["geometry"] = pois_gdf["geometry"].centroid


 Loaded 417 POIs into Spark
+-----------+-----+
|   category|count|
+-----------+-----+
|arts_centre|    1|
|        gym|  215|
|       cafe|    1|
|        pub|  200|
+-----------+-----+



In [5]:
#PHASE 3: Assign everything to H3 hexagons at Resolution 9
import h3
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf, col, avg, count

# --- 3a. Create a PySpark UDF (User Defined Function) for H3
to_h9 = udf(
    lambda lat, lon: h3.latlng_to_cell(float(lat), float(lon), 9)
                     if lat is not None and lon is not None else None,
    StringType()
)

# --- 3b. Assign every Airbnb listing to a hexagon
listings = listings.withColumn("hex_id", to_h9("latitude", "longitude"))

# --- 3c. Assign every POI to a hexagon
poi_spark = poi_spark.withColumn("hex_id", to_h9("lat", "lon"))

# --- 3d. Compute the Affluence Score for each hexagon
hex_price = (
    listings
    .groupBy("hex_id")
    .agg(
        avg("price").alias("affluence_score"),
        count("id").alias("listing_count")
    )
    .filter(col("hex_id").isNotNull())
)

# Force computation to get the total count
total_hexes = hex_price.count()

print(f"Created {total_hexes:,} unique hexagonal neighborhoods!")
print("\nTop 5 Most Expensive Hexagons (Affluence Score):")
hex_price.orderBy(col("affluence_score").desc()).show(5)

Created 3,515 unique hexagonal neighborhoods!

Top 5 Most Expensive Hexagons (Affluence Score):
+---------------+---------------+-------------+
|         hex_id|affluence_score|listing_count|
+---------------+---------------+-------------+
|892a1070b6fffff|         5000.0|            1|
|892a10741d7ffff|         3000.0|            1|
|892a1000bafffff|         2600.0|            1|
|892a10728d3ffff|         2000.0|            2|
|892a1076a93ffff|         2000.0|            1|
+---------------+---------------+-------------+
only showing top 5 rows



In [6]:
# PHASE 4: Sedona spatial join + Gaussian distance decay

import math
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import expr, udf, col, sum as _sum

# --- 4a. Build hex centroid table
get_clat = udf(lambda h: float(h3.cell_to_latlng(h)[0]) if h else None, DoubleType())
get_clon = udf(lambda h: float(h3.cell_to_latlng(h)[1]) if h else None, DoubleType())

hex_centroids = (
    hex_price.select("hex_id")
    .withColumn("c_lat", get_clat("hex_id"))
    .withColumn("c_lon", get_clon("hex_id"))
)

# --- 4b. Project geometry from Degrees (EPSG:4326) to Meters (EPSG:3857)
hex_centroids = hex_centroids.withColumn(
    "centroid_m",
    expr("ST_Transform(ST_Point(CAST(c_lon AS DOUBLE), CAST(c_lat AS DOUBLE)), 'epsg:4326', 'epsg:3857')")
)

poi_spark = poi_spark.withColumn(
    "geom_m",
    expr("ST_Transform(ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE)), 'epsg:4326', 'epsg:3857')")
)

# --- 4c. Register as SQL views so Sedona can run spatial queries on them
hex_centroids.createOrReplaceTempView("hex_centroids")
poi_spark.createOrReplaceTempView("pois")

# --- 4d. Sedona spatial join: ST_DWithin finds all pairs within 1000 meters
nearby = spark.sql("""
    SELECT
        h.hex_id,
        p.poi_weight,
        p.category,
        ST_Distance(h.centroid_m, p.geom_m) AS dist_meters
    FROM hex_centroids h
    JOIN pois p
    ON ST_DWithin(h.centroid_m, p.geom_m, 1000.0)
""")

# --- 4e. Gaussian decay function
BANDWIDTH = 500.0

gaussian_decay = udf(
    lambda dist: math.exp(-(dist ** 2) / (2 * BANDWIDTH ** 2))
                 if dist is not None else 0.0,
    DoubleType()
)

amenity_scores = (
    nearby
    .withColumn("decay", gaussian_decay("dist_meters"))
    .withColumn("contribution", col("decay") * col("poi_weight"))
    .groupBy("hex_id")
    .agg(_sum("contribution").alias("amenity_score"))
)

print(f"Computed distance-decayed amenity scores for {amenity_scores.count():,} hexes")
print("\nTop 5 Most Amenity-Rich Hexagons:")
amenity_scores.orderBy(col("amenity_score").desc()).show(5)

Computed distance-decayed amenity scores for 464 hexes

Top 5 Most Amenity-Rich Hexagons:
+---------------+------------------+
|         hex_id|     amenity_score|
+---------------+------------------+
|892a1072c93ffff|15.235139200348224|
|892a1072c83ffff|14.378103668364378|
|892a1072c97ffff| 14.09673325938643|
|892a100d213ffff| 14.04059393024709|
|892a100d28bffff|13.941616006857663|
+---------------+------------------+
only showing top 5 rows



In [11]:
# PHASE 5: Physical Environment via Dynamic World (Real Public GEE Data)
# We use Google's Dynamic World dataset, which provides a 9-dimensional vector 
# representing the physical land cover of the area (Built, Trees, Grass, etc.).
# We sample this for 2024 and 2017 to measure physical transformation.

import ee
import ee.oauth
import geemap
import pandas as pd

ee.oauth.get_credentials_path = lambda: "/home/jovyan/work/notebooks/gee_credentials.json"

ee.Authenticate(auth_mode='notebook')

ee.Initialize(project="mineral-acrobat-457713-c9") 

centroids_pd = (
    hex_centroids
    .select("hex_id", "c_lat", "c_lon")
    .toPandas()
)

def sample_dynamic_world(year, centroids_pd):
    """Samples the 9-dimensional land cover vector at each hex centroid."""
    
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    
    collection = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(start_date, end_date)
        .select([
            'water', 'trees', 'grass', 'flooded_vegetation', 
            'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'
        ])
        .mean()
    )
    
    # Create GEE points for our hex centroids
    points = ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Point([float(row.c_lon), float(row.c_lat)]),
            {"hex_id": row.hex_id}
        )
        for _, row in centroids_pd.iterrows()
    ])
    
    # Sample the data at our points
    sampled = collection.sampleRegions(
        collection=points, scale=10, geometries=False
    )
    
    # Convert back to a Pandas DataFrame
    features = sampled.getInfo()['features']
    properties_list = [f['properties'] for f in features]
    df = pd.DataFrame(properties_list)
    
    # Return the Hex ID and the 9 physical features
    cols = ['hex_id', 'water', 'trees', 'grass', 'flooded_vegetation', 
            'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice']
    
    # Ensure all expected columns exist
    for col in cols:
        if col not in df.columns:
            df[col] = 0.0
            
    return df[cols]

print("Connecting to Google Earth Engine (Dynamic World)...")
print("Sampling 2024 physical environment (This takes about 1-2 minutes)...")
dw_2024_pd = sample_dynamic_world(2024, centroids_pd)

print("Sampling 2017 historical environment (This takes about 1-2 minutes)...")
dw_2017_pd = sample_dynamic_world(2017, centroids_pd)

print(f"Successfully downloaded real 9-dimensional land cover data for {len(dw_2024_pd)} hexes!")

Connecting to Google Earth Engine (Dynamic World)...
Sampling 2024 physical environment (This takes about 1-2 minutes)...
Sampling 2017 historical environment (This takes about 1-2 minutes)...
Successfully downloaded real 9-dimensional land cover data for 3515 hexes!


In [12]:
# PHASE 6: Geographically Weighted PCA (Dynamic World Edition) 

import numpy as np
import pandas as pd
import h3

# --- 6a. Define the new explainable feature columns
DW_COLS = [
    'water', 'trees', 'grass', 'flooded_vegetation', 
    'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'
]
FEATURE_COLS = ["affluence_score", "amenity_score"] + DW_COLS

BANDWIDTH_M  = 1000.0   # Gaussian kernel bandwidth in metres
K_RING       = 2        # ring size: k=2 means up to 18 surrounding hexes

# --- 6b. Merge all signals into one master hex DataFrame
print("Merging Economic, Social, and Physical data...")
hex_price_pd      = hex_price.toPandas()
amenity_pd        = amenity_scores.toPandas()
centroids_pd_full = hex_centroids.select("hex_id", "c_lat", "c_lon").toPandas()

master_pd = (
    hex_price_pd
    .merge(amenity_pd,        on="hex_id", how="inner")
    .merge(dw_2024_pd,        on="hex_id", how="inner") # Using our new real data!
    .merge(centroids_pd_full, on="hex_id", how="inner")
    .dropna(subset=FEATURE_COLS)
    .reset_index(drop=True)
)

print(f"Master table ready: {len(master_pd)} hexes with all 11 features.")

# --- 6c. Helper: Haversine distance in metres between two lat/lon points
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi   = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# --- 6d. Gaussian kernel weight from distance in metres
def gaussian_weight(dist_m, bandwidth=BANDWIDTH_M):
    return np.exp(-(dist_m**2) / (2 * bandwidth**2))

# --- 6e. GWPCA function for a single hex
def gwpca_for_hex(target_hex_id, master_pd, feature_cols=FEATURE_COLS):
    try:
        target_row  = master_pd[master_pd["hex_id"] == target_hex_id]
        if target_row.empty: return None

        c_lat = target_row.iloc[0]["c_lat"]
        c_lon = target_row.iloc[0]["c_lon"]

        # Get all hexes within k=2 rings
        neighbors = list(h3.grid_disk(target_hex_id, K_RING))
        neighbor_data = master_pd[master_pd["hex_id"].isin(neighbors)].copy()
        
        # We need at least 3 neighbors to find a meaningful mathematical pattern
        if len(neighbor_data) < 3: return None

        # Compute geodesic distances using haversine
        neighbor_data["dist_m"] = neighbor_data.apply(
            lambda r: haversine_m(c_lat, c_lon, r["c_lat"], r["c_lon"]), axis=1
        )

        # Compute kernel weights
        neighbor_data["kernel_w"] = neighbor_data["dist_m"].apply(gaussian_weight)

        X = neighbor_data[feature_cols].values.astype(float)
        W = neighbor_data["kernel_w"].values

        # Weighted mean and centred features
        X_bar = np.average(X, axis=0, weights=W)
        X_c   = X - X_bar

        # Weighted covariance matrix
        cov = (X_c.T * W) @ X_c / W.sum()

        # Eigendecomposition — eigh returns smallest to largest eigenvalue
        eigvals, eigvecs = np.linalg.eigh(cov)
        pc1 = eigvecs[:, -1]   # Dominant local direction

        # Project target hex onto local PC1
        target_features = target_row[feature_cols].values[0].astype(float)
        score = float((target_features - X_bar) @ pc1)
        return score

    except Exception as e:
        return None

# --- 6f. Run GWPCA for all hexes
print("Running Geographically Weighted PCA (Finding local prosperity patterns)...")

hex_ids = master_pd["hex_id"].tolist()
prosperity_scores = []

for i, hid in enumerate(hex_ids):
    score = gwpca_for_hex(hid, master_pd)
    prosperity_scores.append({"hex_id": hid, "local_prosperity": score})
    if i > 0 and i % 100 == 0:
        print(f"  {i}/{len(hex_ids)} hexes processed...")

gwpca_pd = pd.DataFrame(prosperity_scores)

# Normalize scores to 0-100 range for easier interpretation on the map
valid = gwpca_pd["local_prosperity"].dropna()
mn, mx = valid.min(), valid.max()
gwpca_pd["prosperity_normalized"] = (
    (gwpca_pd["local_prosperity"] - mn) / (mx - mn) * 100
)

print(f"\n GWPCA complete! Score range normalized from 0 to 100.")
print("Top 5 Most Locally Prosperous Hexes:")
gwpca_pd.sort_values("prosperity_normalized", ascending=False).head(5)

Merging Economic, Social, and Physical data...
Master table ready: 464 hexes with all 11 features.
Running Geographically Weighted PCA (Finding local prosperity patterns)...
  100/464 hexes processed...
  200/464 hexes processed...
  300/464 hexes processed...
  400/464 hexes processed...

 GWPCA complete! Score range normalized from 0 to 100.
Top 5 Most Locally Prosperous Hexes:


,hex_id,local_prosperity,prosperity_normalized
439,892a10728d3ffff,1417.484011,100.000000
269,892a100d46bffff,1070.084964,81.008947
29,892a100d053ffff,802.510557,66.381620
4,892a10728c3ffff,647.137352,57.887927
182,892a1072dd7ffff,503.189350,50.018809


In [13]:
# PHASE 7: Temporal Drift (Measuring Physical Gentrification)

import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine

print("Calculating physical transformation between 2017 and 2024...")

# 1. Define our physical features
DW_COLS = [
    'water', 'trees', 'grass', 'flooded_vegetation', 
    'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'
]

# 2. Merge the two years of data on hex_id
drift_df = pd.merge(
    dw_2017_pd, 
    dw_2024_pd, 
    on="hex_id", 
    suffixes=("_2017", "_2024")
)

# 3. Calculate Cosine Distance for every hexagon
def calculate_drift(row):
    vec_2017 = row[[f"{c}_2017" for c in DW_COLS]].values.astype(float)
    vec_2024 = row[[f"{c}_2024" for c in DW_COLS]].values.astype(float)
    
    # If a hexagon is completely empty (all zeros), avoid division by zero
    if np.sum(vec_2017) == 0 or np.sum(vec_2024) == 0:
        return 0.0
        
    return cosine(vec_2017, vec_2024)

drift_df["temporal_drift"] = drift_df.apply(calculate_drift, axis=1)

# 4. Normalize the drift score to a 0-100 scale for intuitive mapping
d_min = drift_df["temporal_drift"].min()
d_max = drift_df["temporal_drift"].max()

if d_max > d_min:
    drift_df["drift_normalized"] = ((drift_df["temporal_drift"] - d_min) / (d_max - d_min)) * 100
else:
    drift_df["drift_normalized"] = 0.0

print(f"Calculated Temporal Drift for {len(drift_df)} neighborhoods!")

# 5. Show the neighborhoods that changed the most
print("\nTop 5 Neighborhoods with the Most Physical Transformation (2017 -> 2024):")
top_drift = drift_df.sort_values("drift_normalized", ascending=False).head(5)

# We display the built and trees columns so you can physically see the change!
display_cols = ["hex_id", "drift_normalized", "built_2017", "built_2024", "trees_2017", "trees_2024"]
print(top_drift[display_cols].to_string(index=False))

Calculating physical transformation between 2017 and 2024...
Calculated Temporal Drift for 3515 neighborhoods!

Top 5 Neighborhoods with the Most Physical Transformation (2017 -> 2024):
         hex_id  drift_normalized  built_2017  built_2024  trees_2017  trees_2024
892a10012cfffff        100.000000    0.049850    0.127168    0.054305    0.048513
892a1039c13ffff         66.751567    0.080053    0.085056    0.028509    0.021731
892a1062dcfffff         65.128434    0.143313    0.282118    0.339708    0.092280
892a107656bffff         52.669872    0.334531    0.108526    0.136040    0.323946
892a103b413ffff         43.169176    0.205161    0.423468    0.374223    0.157462


In [19]:
# PHASE 8: Interactive Mapping 

import folium
import branca.colormap as cm
import pandas as pd
import h3

print("Generating Multi-Layer Map...")

map_data = pd.merge(gwpca_pd, drift_df, on="hex_id", how="inner")


m = folium.Map(location=[40.7831, -73.9712], zoom_start=12, tiles="CartoDB positron")


prosper_cmap = cm.LinearColormap(
    colors=['#bae4b3', '#74c476', '#31a354', '#006d2c', '#00441b'], 
    vmin=0, vmax=100, 
    caption='Local Prosperity Score (0-100)'
)

# Drift: Dropped the peach, ends in a deep crimson red
drift_cmap = cm.LinearColormap(
    colors=['#fcbba1', '#fc9272', '#fb6a4a', '#ef3b2c', '#cb181d', '#99000d'], 
    vmin=0, vmax=100, 
    caption='Physical Transformation / Drift Score (0-100)'
)

fg_prosper = folium.FeatureGroup(name="1. Local Prosperity (Green)", show=True)
fg_drift   = folium.FeatureGroup(name="2. Physical Transformation (Red)", show=False)

for _, row in map_data.iterrows():
    hid = row["hex_id"]
    p_score = row["prosperity_normalized"]
    d_score = row["drift_normalized"]
    
    boundary = h3.cell_to_boundary(hid)
    
    # --- PROSPERITY POLYGON ---
    p_text = f"<b>Prosperity:</b> {p_score:.1f}/100"
    folium.Polygon(
        locations=boundary,
        color=prosper_cmap(p_score),
        weight=1.5,                       
        fill=True,
        fill_color=prosper_cmap(p_score),
        fill_opacity=0.5,                 
        tooltip=p_text
    ).add_to(fg_prosper)
    
    # --- DRIFT POLYGON ---
    b_17, b_24 = row["built_2017"], row["built_2024"]
    t_17, t_24 = row["trees_2017"], row["trees_2024"]
    
    d_text = (
        f"<b>Transformation Drift:</b> {d_score:.1f}/100<br>"
        f"<b>Built:</b> {b_17:.0%} → {b_24:.0%}<br>"
        f"<b>Trees:</b> {t_17:.0%} → {t_24:.0%}"
    )
    folium.Polygon(
        locations=boundary,
        color=drift_cmap(d_score),
        weight=1.5,                       
        fill=True,
        fill_color=drift_cmap(d_score),
        fill_opacity=0.5,                 
        tooltip=d_text
    ).add_to(fg_drift)

fg_prosper.add_to(m)
fg_drift.add_to(m)

m.add_child(prosper_cmap)
m.add_child(drift_cmap)
folium.LayerControl(collapsed=False).add_to(m)

print("Map generation complete!")
m.save("index.html")
m

Generating Multi-Layer Map...
Map generation complete!


In [ ]:
# PHASE 8B: Top 10 Highlight Map 

import folium
import h3

print("Generating Top 10 Highlight Map...")

m_top = folium.Map(location=[40.7831, -73.9712], zoom_start=12, tiles="CartoDB positron")

# Filter for Top 10 in both categories
top_10_prosper = map_data.sort_values("prosperity_normalized", ascending=False).head(10)
top_10_drift = map_data.sort_values("drift_normalized", ascending=False).head(10)

fg_top_prosper = folium.FeatureGroup(name="1. Top 10 Local Prosperity", show=True)
fg_top_drift   = folium.FeatureGroup(name="2. Top 10 Physical Transformation", show=False)

# --- Add Top 10 Prosperity Polygons ---
for _, row in top_10_prosper.iterrows():
    hid = row["hex_id"]
    p_score = row["prosperity_normalized"]
    boundary = h3.cell_to_boundary(hid)
    
    p_text = f"<b>Top 10 Prosperity</b><br>Score: {p_score:.1f}/100"
    folium.Polygon(
        locations=boundary,
        color="#00441b",          # Static dark green for Top 10
        weight=2,                       
        fill=True,
        fill_color="#006d2c",
        fill_opacity=0.5,         # 50% transparency
        tooltip=p_text
    ).add_to(fg_top_prosper)

# --- Add Top 10 Drift Polygons ---
for _, row in top_10_drift.iterrows():
    hid = row["hex_id"]
    d_score = row["drift_normalized"]
    b_17, b_24 = row["built_2017"], row["built_2024"]
    t_17, t_24 = row["trees_2017"], row["trees_2024"]
    
    boundary = h3.cell_to_boundary(hid)
    
    d_text = (
        f"<b>Top 10 Transformation</b><br>Drift Score: {d_score:.1f}/100<br>"
        f"<b>Built:</b> {b_17:.0%} → {b_24:.0%}<br>"
        f"<b>Trees:</b> {t_17:.0%} → {t_24:.0%}"
    )
    folium.Polygon(
        locations=boundary,
        color="#99000d",          # Static dark red for Top 10
        weight=2,                       
        fill=True,
        fill_color="#cb181d",
        fill_opacity=0.5,         # 50% transparency
        tooltip=d_text
    ).add_to(fg_top_drift)

fg_top_prosper.add_to(m_top)
fg_top_drift.add_to(m_top)

folium.LayerControl(collapsed=False).add_to(m_top)

print("Top 10 Map generation complete!")
m_top.save("index_top10.html")
m_top